# Advanced Extension — QA Subgraph

This notebook implements the optional **Advanced Extension** from the project brief: instead of a single `qa_node` doing everything in one LLM call, QA becomes its own internal pipeline — a `QA_Subgraph` — that routes between:

1. **`syntax_checker_node`** — a plain Python check (no LLM, no cost) that catches import/syntax errors immediately.
2. **`logic_tester_node`** — an LLM call that reasons about functional bugs and missing features (only runs if syntax passed — no point analyzing logic in code that can't even parse).
3. **`performance_auditor_node`** — the scorer: turns whatever the subgraph found into the final 1–10 score.

**The key requirement from the brief:** this entire subgraph must appear as **one single node** to the parent graph. LangGraph supports this natively — a *compiled* `StateGraph` is itself callable, so you can pass the compiled subgraph object straight into `parent_graph.add_node("qa", compiled_subgraph)`. The parent graph never sees `syntax_checker_node`, `logic_tester_node`, or `performance_auditor_node` individually — it only sees one node called `qa` that goes in with code and comes out with a score.

## Why split QA into three nodes instead of one?

The original single `qa_node` asked one LLM call to catch everything at once — syntax problems, logic bugs, and missing features — all in the same prompt. That works, but it has two weaknesses this subgraph fixes:

- **Wasted LLM calls on broken code.** If the Engineer's code has a syntax error, there's nothing to reason about — `ast.parse` catches that in milliseconds for free. No need to spend an LLM call diagnosing logic in code that can't run.
- **Mixing concerns lowers precision.** A single prompt juggling "is this valid Python?" and "does this correctly implement ducking?" tends to blur both. Splitting them lets each node specialize: syntax checking is deterministic and exact, logic testing is where the LLM's reasoning is actually needed.

This mirrors how a real QA pipeline works: a linter runs before a human ever reviews the logic.

In [0]:
# Same environment as the main notebook — this subgraph is meant to be
# pasted into that notebook (or imported from a shared module) so it can
# reuse `llm`, `get_text`, and the message classes already defined there.
import ast
import re
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, END

# If running this notebook standalone (not pasted into the main one),
# uncomment the lines below to redefine what it depends on:

# import os
# from langchain_google_genai import ChatGoogleGenerativeAI
# os.environ["GOOGLE_API_KEY"] = dbutils.secrets.get(scope="dino-runner-secrets", key="google-api-key")
# llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.2)
#
# def get_text(message) -> str:
#     content = message.content
#     if isinstance(content, str):
#         return content
#     if isinstance(content, list):
#         return "".join(part.get("text", "") for part in content if isinstance(part, dict))
#     return str(content)

## Subgraph State

A subgraph's state schema only needs the **keys it actually reads and writes**. LangGraph matches keys by name against the parent's `GameState` when the subgraph is added as a node, so as long as the field names line up (`engineer_code`, `architect_messages`, `director_messages`, `qa_feedback`, `iteration_score`), values pass through automatically in both directions — no manual translation layer needed.

Two fields are internal to the subgraph only and never touch the parent: `syntax_ok` and `syntax_error`, used purely to route between the three internal nodes.

In [0]:
import operator

class QAState(TypedDict):
    # Shared with the parent GameState — same names, so values flow
    # through automatically when this subgraph is plugged in as a node.
    engineer_code: Annotated[list, add_messages]
    architect_messages: Annotated[list, add_messages]
    director_messages: Annotated[list, add_messages]
    qa_feedback: Annotated[list, add_messages]
    iteration_score: Annotated[list[int], operator.add]

    # Internal-only routing fields — never read by the parent graph.
    syntax_ok: bool
    syntax_error: str

## Node 1 — Syntax Checker (no LLM)

Pure Python: attempts to parse the Engineer's code with `ast.parse`. This is deterministic, instant, and free — a real syntax error will crash `subprocess.run` in the parent's execution step anyway, so catching it here first saves an LLM round-trip on code that's already known to be broken.

In [0]:
def syntax_checker_node(state: QAState):
    code = get_text(state["engineer_code"][-1])
    try:
        ast.parse(code)
        return {"syntax_ok": True, "syntax_error": ""}
    except SyntaxError as e:
        return {"syntax_ok": False, "syntax_error": f"{e.msg} (line {e.lineno})"}

## Node 2 — Logic Tester (LLM)

Only reached if the syntax check passed. This is essentially the diagnostic half of the original `qa_node` — checking the code and execution log against the Director's original requirements and the Architect's design — but it no longer has to worry about whether the file even parses.

In [0]:
def logic_tester_node(state: QAState):
    code = get_text(state["engineer_code"][-1])
    original_requirements = get_text(state["director_messages"][-1])
    design = get_text(state["architect_messages"][-1])

    system_prompt = (
        "You are a logic-focused QA engineer. The code has already been "
        "confirmed to be syntactically valid Python — do not comment on "
        "syntax. Your job is purely functional: compare the code against "
        "the original requirements and design, and list concrete logic "
        "bugs or missing features. Explicitly check for: flying obstacles, "
        "ground obstacles, jump/fall physics, a distinct duck mechanic, and "
        "a working high-score tracker. Do NOT write corrected code — "
        "diagnosis only. If the code is functionally correct, say so "
        "explicitly and report fewer than 2 issues rather than inventing any."
    )

    instruction = (
        f"Original requirements:\n{original_requirements}\n\n"
        f"Design:\n{design}\n\n"
        f"Code:\n{code}"
    )

    response = llm.invoke([
        ("system", system_prompt),
        ("human", instruction)
    ])

    return {"qa_feedback": [AIMessage(content=get_text(response))]}

## Node 3 — Performance Auditor (the subgraph's Scorer)

The brief names this node explicitly as the scorer. It has two possible paths in:

- **From the syntax checker directly** (syntax failed) — an automatic 1–3 score, no LLM call needed, since broken syntax always means the lowest band.
- **From the logic tester** (syntax passed) — reads the logic report and scores it with the same 1–10 banded rubric as before.

In [0]:
def performance_auditor_node(state: QAState):
    # Path A: syntax failed upstream — score without an LLM call.
    if not state["syntax_ok"]:
        note = f"Code failed to parse: {state['syntax_error']}"
        return {
            "qa_feedback": [AIMessage(content=note)],
            "iteration_score": [1]
        }

    # Path B: syntax passed, logic_tester already appended its report.
    qa_report = get_text(state["qa_feedback"][-1])

    system_prompt = (
        "You are a scoring engine. Read the QA report and output ONLY "
        "a single integer from 1 to 10 representing overall code health "
        "and feature completeness. No words, no explanation, just the number.\n\n"
        "Scoring bands:\n"
        "9-10: No meaningful bugs remain, all requirements are met.\n"
        "7-8: Minor cosmetic or edge-case issues only.\n"
        "4-6: At least one functional bug, but the game is playable.\n"
        "1-3: Missing a core required feature (jumping, ducking, obstacles, or scoring).\n\n"
        "Do not default to the middle of the range — use the full scale."
    )

    response = llm.invoke([
        ("system", system_prompt),
        ("human", qa_report)
    ])

    raw = get_text(response).strip()
    match = re.search(r"\d+", raw)
    score = int(match.group()) if match else 5
    score = max(1, min(10, score))

    return {"iteration_score": [score]}

## Wiring the Subgraph

Routing logic: `syntax_checker` → if syntax failed, go **straight** to `performance_auditor` (skip the wasted LLM call entirely); if syntax passed, go through `logic_tester` first.

In [0]:
qa_subgraph_builder = StateGraph(QAState)

qa_subgraph_builder.add_node("syntax_checker", syntax_checker_node)
qa_subgraph_builder.add_node("logic_tester", logic_tester_node)
qa_subgraph_builder.add_node("performance_auditor", performance_auditor_node)

qa_subgraph_builder.set_entry_point("syntax_checker")

def route_after_syntax(state: QAState):
    return "logic_tester" if state["syntax_ok"] else "performance_auditor"

qa_subgraph_builder.add_conditional_edges(
    "syntax_checker",
    route_after_syntax,
    {
        "logic_tester": "logic_tester",
        "performance_auditor": "performance_auditor"
    }
)

qa_subgraph_builder.add_edge("logic_tester", "performance_auditor")
qa_subgraph_builder.add_edge("performance_auditor", END)

# Compiled WITHOUT its own checkpointer - it shares the parent graph's
# checkpointer/thread automatically when added as a node below. Giving
# it a separate MemorySaver here would fork its state history away from
# the parent run.
qa_subgraph = qa_subgraph_builder.compile()

## Plugging It Into the Parent Graph

This is the part that satisfies the brief's requirement — *"this entire complex graph must appear as a single node to the parent graph."* You don't call `qa_subgraph.invoke(...)` from inside a wrapper function; you hand the **compiled subgraph object itself** to `add_node`. LangGraph treats it exactly like any other node function: the parent graph calls it once, it internally runs its full syntax → logic → score sequence, and returns the merged state — the parent never sees the three inner steps.

To use this in the main notebook, replace the two lines that add `"qa"` and `"scorer"` as separate nodes/edges with the single node below, and update the conditional edge after it to route from `"qa"` (not `"scorer"`).

In [0]:
# --- In the main notebook's graph-building cell, replace this: ---
#
#   graph.add_node("qa", qa_node)
#   graph.add_node("scorer", scorer_node)
#   graph.add_edge("execution", "qa")
#   graph.add_edge("qa", "scorer")
#   graph.add_conditional_edges("scorer", route_after_scoring, {...})
#
# --- with this: ---
#
#   graph.add_node("qa", qa_subgraph)          # <- the whole subgraph, one node
#   graph.add_edge("execution", "qa")
#   graph.add_conditional_edges("qa", route_after_scoring, {...})
#
# Everything else in the parent graph (director, architect, engineer,
# save_code, execution, the human-in-the-loop routing, GameState,
# MemorySaver) stays exactly the same. The parent graph's `GameState`
# doesn't need `syntax_ok` / `syntax_error` added to it — those are
# local to the subgraph and simply don't propagate outward.

## Quick Standalone Test

Run the subgraph directly (outside the parent graph) against a deliberately broken snippet, to confirm the syntax-failure shortcut works without touching the LLM.

In [0]:
test_state = {
    "engineer_code": [AIMessage(content="def broken(:\n    pass")],  # invalid syntax on purpose
    "architect_messages": [AIMessage(content="Player class, obstacle spawner, scoring.")],
    "director_messages": [HumanMessage(content="Build a Dino Runner game.")],
    "qa_feedback": [],
    "iteration_score": [],
    "syntax_ok": True,
    "syntax_error": ""
}

result = qa_subgraph.invoke(test_state)
print("Score:", result["iteration_score"])
print("Feedback:", get_text(result["qa_feedback"][-1]))